In [5]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT     = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
DATA_DIR = ROOT / "Data"

TRAIN_MERGED = DATA_DIR / "X_train_merged.csv"
TEST_MERGED  = DATA_DIR / "X_test_merged.csv"
Y_PATH       = ROOT / "y_train.csv"

N_SPLITS      = 5
SEED          = 42
PCA_CLIP_DIM  = 128   # dimensions PCA pour embeddings CLIP  (c_*)
PCA_VGG_DIM   = 64   # dimensions PCA pour embeddings VGGish (vgg_*)

# =========================
# UTILS
# =========================
def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        try:
            return pd.read_csv(path, sep=";", engine="python")
        except Exception:
            return pd.read_csv(path, sep=",", engine="python")

def normalize_id_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"^(VIDEO_|video_|Video_)", "", regex=True)
    s = s.str.extract(r"(\d+)", expand=False).fillna(s)
    return s.str.strip()

# =========================
# 1) LOAD DATA
# =========================
print("📂 Chargement des données...")
train = read_csv_robust(TRAIN_MERGED)
test  = read_csv_robust(TEST_MERGED)
y_df  = read_csv_robust(Y_PATH)

# Normalize target column name
if "ID" not in y_df.columns:
    for candidate in ["video_id", "id"]:
        if candidate in y_df.columns:
            y_df = y_df.rename(columns={candidate: "ID"})
            break
    else:
        y_df = y_df.rename(columns={y_df.columns[0]: "ID"})

# Normalize ID columns
for df in (train, test, y_df):
    if "ID" not in df.columns:
        for candidate in ["video_id", "id"]:
            if candidate in df.columns:
                df.rename(columns={candidate: "ID"}, inplace=True)
                break
        else:
            raise ValueError(f"Colonne ID introuvable: {df.columns.tolist()[:20]}")
    df["ID"] = normalize_id_series(df["ID"])

# Drop popularity si déjà présent (éviter leakage)
for df in (train, test):
    if "popularity" in df.columns:
        df.drop(columns=["popularity"], inplace=True)

# Merge target
train = train.merge(y_df[["ID", "popularity"]], on="ID", how="left", validate="m:1")
if train["popularity"].isna().any():
    n_miss = train["popularity"].isna().sum()
    print(f"  ⚠️  {n_miss} IDs sans label → supprimés")
    train = train.dropna(subset=["popularity"]).reset_index(drop=True)

print(f"  Train : {train.shape}  |  Test : {test.shape}")

# =========================
# 2) ALIGN COLUMNS
# =========================
drop_cols = ["ID", "popularity"]
train_cols = set(train.columns) - set(drop_cols)
test_cols  = set(test.columns) - {"ID"}

for c in sorted(train_cols - test_cols):
    test[c] = np.nan
for c in sorted(test_cols - train_cols):
    train[c] = np.nan

feature_candidates = sorted(list((set(train.columns) - set(drop_cols))))
train = train[["ID"] + feature_candidates + ["popularity"]]
test  = test[["ID"]  + feature_candidates]

# =========================
# 3) FEATURE REDUCTION (redondances manuelles)
# =========================
print("\n✂️  Réduction des features redondantes...")
cols_before = len(feature_candidates)

# -- A) MFCCs : garder seulement 0-5 (mean + std), dropper 6-12
mfcc_to_drop = []
for i in range(6, 20):  # sécurité jusqu'à 20
    for stat in ("mean", "std"):
        c = f"mfcc_{i}_{stat}"
        if c in feature_candidates:
            mfcc_to_drop.append(c)
print(f"   MFCCs 6+ droppés      : {len(mfcc_to_drop)}  ({mfcc_to_drop[:4]}{'...' if len(mfcc_to_drop)>4 else ''})")

# -- B) Géométrie vidéo : garder resolution_area + aspect, dropper width & height
geo_to_drop = [c for c in ["width", "height"] if c in feature_candidates]
print(f"   Géo redondante droppée : {geo_to_drop}")

# -- C) Counts bruts : garder xxx_log_count, dropper xxx_count (non-log)
#    Pattern : si 'xxx_count' ET 'xxx_log_count' existent → drop 'xxx_count'
count_to_drop = []
for c in feature_candidates:
    if re.fullmatch(r".+_count", c):          # ex: uploader_count
        log_version = c.replace("_count", "_log_count")
        if log_version in feature_candidates:
            count_to_drop.append(c)
print(f"   Counts bruts droppés   : {len(count_to_drop)}  ({count_to_drop})")

# -- Appliquer les drops sur les deux splits
all_to_drop = set(mfcc_to_drop + geo_to_drop + count_to_drop)
feature_candidates = [c for c in feature_candidates if c not in all_to_drop]
train = train[["ID"] + feature_candidates + ["popularity"]]
test  = test[["ID"]  + feature_candidates]

print(f"   Features : {cols_before} → {len(feature_candidates)}  (−{cols_before - len(feature_candidates)})")

# =========================
# 4) DETECT EMBEDDING COLUMNS
# =========================
# VGGish : préfixe 'vgg'
# CLIP   : préfixe 'c_' ou 'clip_'
vgg_cols  = [c for c in feature_candidates if c.lower().startswith("vgg")]
clip_cols = [c for c in feature_candidates if c.lower().startswith("c") or c.lower().startswith("clip_")]
non_numeric = train[feature_candidates].select_dtypes(exclude=[np.number]).columns.tolist()
other_cols  = [c for c in feature_candidates
               if c not in vgg_cols and c not in clip_cols and c not in non_numeric]

print(f"\n🔍 Embeddings détectés :")
print(f"   VGGish : {len(vgg_cols)} colonnes  (ex: {vgg_cols[:3]})")
print(f"   CLIP   : {len(clip_cols)} colonnes  (ex: {clip_cols[:3]})")
print(f"   Autres features numériques : {len(other_cols)}")
print(f"   Non-numériques droppées    : {len(non_numeric)}")

# =========================
# 4) PCA SUR LES EMBEDDINGS
# =========================
def apply_pca(train_df, test_df, cols, n_components, name):
    if len(cols) == 0:
        print(f"  ⚠️  Aucune colonne {name} trouvée, PCA ignorée.")
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)

    n_comp = min(n_components, len(cols), len(train_df))

    X_tr = train_df[cols].fillna(0).values
    X_te = test_df[cols].fillna(0).values

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    pca = PCA(n_components=n_comp, random_state=SEED)
    tr_pca = pca.fit_transform(X_tr_s)
    te_pca = pca.transform(X_te_s)

    var_explained = pca.explained_variance_ratio_.cumsum()[-1]
    print(f"  ✅ PCA {name}: {len(cols)} → {n_comp} dims  |  variance expliquée : {var_explained:.1%}")

    col_names = [f"{name}_pca_{i}" for i in range(n_comp)]
    return (
        pd.DataFrame(tr_pca, columns=col_names, index=train_df.index),
        pd.DataFrame(te_pca, columns=col_names, index=test_df.index),
    )

print("\n🔧 Application de la PCA...")
train_vgg_pca,  test_vgg_pca  = apply_pca(train, test, vgg_cols,  PCA_VGG_DIM,  "vgg")
train_clip_pca, test_clip_pca = apply_pca(train, test, clip_cols, PCA_CLIP_DIM, "clip")

# =========================
# 5) BUILD FINAL FEATURE MATRIX
# =========================
X = pd.concat([
    train[other_cols].reset_index(drop=True),
    train_vgg_pca.reset_index(drop=True),
    train_clip_pca.reset_index(drop=True),
], axis=1)

X_test_final = pd.concat([
    test[other_cols].reset_index(drop=True),
    test_vgg_pca.reset_index(drop=True),
    test_clip_pca.reset_index(drop=True),
], axis=1)

y = train["popularity"].astype(float).values
feature_cols = X.columns.tolist()

print(f"\n📊 Features finales : {len(feature_cols)}")
print(f"   NaN dans X       : {int(X.isna().sum().sum())}")
print(f"   NaN dans X_test  : {int(X_test_final.isna().sum().sum())}")

# =========================
# 6) LIGHTGBM PARAMS
# =========================
lgb_params = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        31,          # bas → évite overfitting
    "max_depth":         6,
    "min_child_samples": 30,          # régularisation forte
    "feature_fraction":  0.5,         # clé avec beaucoup de features
    "bagging_fraction":  0.7,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "verbose":           -1,
    "random_state":      SEED,
}

# =========================
# 7) KFOLD CV
# =========================
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds   = np.zeros(len(X))
test_preds  = np.zeros(len(X_test_final))
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*55}")
print(f"  KFold CV — {N_SPLITS} folds  |  LightGBM")
print(f"{'='*55}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = lgb.LGBMRegressor(n_estimators=2000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=300),
        ]
    )

    oof_preds[val_idx]  = model.predict(X_val)
    test_preds         += model.predict(X_test_final) / N_SPLITS
    feature_imp        += model.feature_importances_ / N_SPLITS

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold} | Best iter : {model.best_iteration_:4d} | RMSE : {fold_rmse:.4f}")

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n{'='*55}")
print(f"  ✅ OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*55}")

# =========================
# 8) TOP FEATURES
# =========================
imp_df = (
    pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
    .sort_values("importance", ascending=False)
)
print("\n🏆 Top 20 features:")
print(imp_df.head(20).to_string(index=False))

imp_df.to_csv(ROOT / "feature_importances.csv", index=False)

# =========================
# 9) SUBMISSION
# =========================
submission = pd.DataFrame({
    "ID":         test["ID"].astype(str).values,
    "popularity": test_preds,
})

assert submission["ID"].isna().sum() == 0,         "❌ IDs manquants !"
assert submission["popularity"].isna().sum() == 0, "❌ Prédictions NaN !"

out_path = ROOT / "submission_lgbm_pca.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print("\nAperçu:")
print(submission.head(10).to_string(index=False))
print("\nStats popularity prédite:")
print(submission["popularity"].describe().round(4))

📂 Chargement des données...
  Train : (1348, 951)  |  Test : (338, 950)

✂️  Réduction des features redondantes...
   MFCCs 6+ droppés      : 14  (['mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std']...)
   Géo redondante droppée : ['width', 'height']
   Counts bruts droppés   : 7  (['album_count', 'artist_count', 'channel_count', 'track_count', 'uid_count', 'uploader_count', 'uploader_short_count'])
   Features : 949 → 926  (−23)

🔍 Embeddings détectés :
   VGGish : 256 colonnes  (ex: ['vggish_000', 'vggish_001', 'vggish_002'])
   CLIP   : 518 colonnes  (ex: ['c0', 'c1', 'c10'])
   Autres features numériques : 144
   Non-numériques droppées    : 8

🔧 Application de la PCA...
  ✅ PCA vgg: 256 → 64 dims  |  variance expliquée : 84.1%
  ✅ PCA clip: 518 → 128 dims  |  variance expliquée : 77.8%

📊 Features finales : 336
   NaN dans X       : 7586
   NaN dans X_test  : 1881

  KFold CV — 5 folds  |  LightGBM
  Fold 1 | Best iter :  128 | RMSE : 1.2481
[300]	valid_0's rmse: 1.3225
  F

In [6]:
import pandas as pd
df = pd.read_csv("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity/Data/X_train_merged.csv", nrows=0)
print(list(df.columns))

['ID', 'f1_sharpness', 'f1_brightness', 'f1_saturation', 'f2_sharpness', 'f2_brightness', 'f2_saturation', 'f3_sharpness', 'f3_brightness', 'f3_saturation', 'f4_sharpness', 'f4_brightness', 'f4_saturation', 'f5_sharpness', 'f5_brightness', 'f5_saturation', 'hook_motion', 'hook_shake', 'content_motion', 'content_shake', 'max_peak_motion', 'yolo_person', 'yolo_ski_snow', 'c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24', 'c25', 'c26', 'c27', 'c28', 'c29', 'c30', 'c31', 'c32', 'c33', 'c34', 'c35', 'c36', 'c37', 'c38', 'c39', 'c40', 'c41', 'c42', 'c43', 'c44', 'c45', 'c46', 'c47', 'c48', 'c49', 'c50', 'c51', 'c52', 'c53', 'c54', 'c55', 'c56', 'c57', 'c58', 'c59', 'c60', 'c61', 'c62', 'c63', 'c64', 'c65', 'c66', 'c67', 'c68', 'c69', 'c70', 'c71', 'c72', 'c73', 'c74', 'c75', 'c76', 'c77', 'c78', 'c79', 'c80', 'c81', 'c82', 'c83', 'c84', 'c85', 'c86', 'c87', 'c88', 'c89', 'c90', '